# Credit Card Fraud Detection - Production-Ready ML Pipeline

## Business Objective
Build a fraud detection system that minimizes financial loss while maintaining acceptable customer experience. This notebook addresses the highly imbalanced classification problem using state-of-the-art sampling techniques and ensemble methods.

## Key Considerations
- **Class Imbalance**: Fraud cases represent <1% of transactions
- **Cost Asymmetry**: False negatives (missed fraud) are far more costly than false positives
- **Business Impact**: Model performance directly affects financial loss and customer satisfaction

---

## Table of Contents
1. [Setup & Imports](#setup)
2. [Data Loading & EDA](#eda)
3. [Data Preprocessing](#preprocessing)
4. [Handling Class Imbalance](#imbalance)
5. [Model Training](#modeling)
6. [Model Evaluation](#evaluation)
7. [Business Interpretation](#business)

<a id='setup'></a>
## 1. Setup & Imports

In [ ]:
# Core Data Science Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    precision_recall_curve, roc_curve, f1_score,
    precision_score, recall_score
)

# Sampling Techniques
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler, SMOTE

# Models
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Warnings
import warnings
warnings.filterwarnings('ignore')

# Plotting Configuration
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

<a id='eda'></a>
## 2. Data Loading & Exploratory Data Analysis (EDA)

In [ ]:
# Load the dataset
df = pd.read_csv('../creditcard.csv')

print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"\nShape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nMemory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

In [ ]:
# Display first few rows
print("\nFirst 5 Rows:")
df.head()

In [ ]:
# Check data types and missing values
print("\n" + "=" * 60)
print("DATA QUALITY CHECK")
print("=" * 60)

missing_info = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing %': (df.isnull().sum() / len(df) * 100).round(2),
    'Data Type': df.dtypes
})
print(missing_info[missing_info['Missing Count'] > 0] if missing_info['Missing Count'].sum() > 0 else "\n✓ No missing values detected!")

In [ ]:
# Class Distribution Analysis
print("\n" + "=" * 60)
print("CLASS DISTRIBUTION")
print("=" * 60)

class_counts = df['Class'].value_counts()
class_pct = df['Class'].value_counts(normalize=True) * 100

class_summary = pd.DataFrame({
    'Count': class_counts,
    'Percentage': class_pct.round(2)
})
class_summary.index = ['Legitimate (0)', 'Fraud (1)']
print(class_summary)
print(f"\nImbalance Ratio: {class_counts[0]/class_counts[1]:.0f}:1")

In [ ]:
# Visualize Class Imbalance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count Plot
ax1 = sns.countplot(x='Class', data=df, ax=axes[0], palette=['#2ecc71', '#e74c3c'])
axes[0].set_title('Class Distribution (Count)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Class (0=Legitimate, 1=Fraud)')
axes[0].set_ylabel('Count')
for p in ax1.patches:
    ax1.annotate(f'{p.get_height():,}', (p.get_x() + p.get_width()/2., p.get_height()),
                 ha='center', va='bottom', fontsize=11)

# Pie Chart
colors = ['#2ecc71', '#e74c3c']
axes[1].pie(class_counts.values, labels=['Legitimate', 'Fraud'], autopct='%1.2f%%',
            colors=colors, explode=[0, 0.1], shadow=True, startangle=90)
axes[1].set_title('Class Distribution (Percentage)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('class_imbalance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Statistical Summary by Class
print("\n" + "=" * 60)
print("STATISTICAL SUMMARY BY CLASS")
print("=" * 60)

numeric_cols = [col for col in df.columns if col not in ['Class']]
print(f"\nAnalyzing {len(numeric_cols)} numeric features...")

# Key statistics for Amount feature (often important in fraud detection)
print("\n--- Amount Feature Analysis ---")
amount_stats = df.groupby('Class')['Amount'].agg(['count', 'mean', 'std', 'min', 'median', 'max'])
amount_stats.columns = ['Count', 'Mean', 'Std', 'Min', 'Median', 'Max']
amount_stats.index = ['Legitimate', 'Fraud']
print(amount_stats.round(2))

In [ ]:
# Distribution of Amount by Class
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot
sns.boxplot(x='Class', y='Amount', data=df, ax=axes[0], palette=['#2ecc71', '#e74c3c'])
axes[0].set_title('Transaction Amount by Class', fontsize=14, fontweight='bold')
axes[0].set_yscale('log')  # Log scale due to outliers

# Histogram
df[df['Class'] == 0]['Amount'].hist(bins=100, alpha=0.5, label='Legitimate', ax=axes[1], color='#2ecc71')
df[df['Class'] == 1]['Amount'].hist(bins=50, alpha=0.5, label='Fraud', ax=axes[1], color='#e74c3c')
axes[1].set_title('Transaction Amount Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Amount')
axes[1].set_ylabel('Frequency')
axes[1].legend()
axes[1].set_xscale('log')

plt.tight_layout()
plt.savefig('amount_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation Analysis
print("\n" + "=" * 60)
print("CORRELATION ANALYSIS")
print("=" * 60)

# Calculate correlations with target
corr_with_target = df.corr()['Class'].drop('Class').abs().sort_values(ascending=False)
print("\nTop 10 Features Most Correlated with Fraud:")
print(corr_with_target.head(10).round(3))

In [ ]:
# Heatmap of correlations (subset of features for clarity)
plt.figure(figsize=(14, 10))

# Select subset of features for visualization
features_for_heatmap = ['Amount'] + [f'V{i}' for i in range(1, 15)] + ['Class']
corr_matrix = df[features_for_heatmap].corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=False, cmap='coolwarm', center=0,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

### EDA Key Findings:
- **Severe Class Imbalance**: Fraud cases represent ~0.17% of transactions (ratio ~570:1)
- **No Missing Values**: Dataset is complete
- **Amount Distribution**: Fraudulent transactions tend to have different amount patterns
- **PCA Features**: V1-V28 are anonymized PCA components (privacy protection)

---

<a id='preprocessing'></a>
## 3. Data Preprocessing

In [ ]:
# Separate features and target
X = df.drop('Class', axis=1)
y = df['Class']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

In [ ]:
# Stratified Train-Test Split
# Stratification ensures class distribution is preserved in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y  # Critical for imbalanced datasets
)

print("\n" + "=" * 60)
print("TRAIN-TEST SPLIT RESULTS")
print("=" * 60)
print(f"\nTraining Set: {X_train.shape[0]:,} samples")
print(f"  - Fraud cases: {y_train.sum():,} ({y_train.mean()*100:.2f}%)")
print(f"\nTest Set: {X_test.shape[0]:,} samples")
print(f"  - Fraud cases: {y_test.sum():,} ({y_test.mean()*100:.2f}%)")

In [ ]:
# Feature Scaling
# Note: V1-V28 are already scaled (PCA components), but Amount needs scaling
scaler = StandardScaler()

# Scale only the 'Amount' feature (V1-V28 are already normalized from PCA)
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled['Scaled_Amount'] = scaler.fit_transform(X_train[['Amount']])
X_test_scaled['Scaled_Amount'] = scaler.transform(X_test[['Amount']])

# Drop original Amount column
X_train_scaled = X_train_scaled.drop('Amount', axis=1)
X_test_scaled = X_test_scaled.drop('Amount', axis=1)

print("\n✓ Feature scaling applied to 'Amount' column")
print(f"Scaled features shape: {X_train_scaled.shape}")

<a id='imbalance'></a>
## 4. Handling Class Imbalance

In [ ]:
# Define sampling strategies
print("\n" + "=" * 60)
print("SAMPLING STRATEGIES OVERVIEW")
print("=" * 60)

# 1. Undersampling - Reduce majority class
rus = RandomUnderSampler(random_state=42)
X_rus, y_rus = rus.fit_resample(X_train_scaled, y_train)
print(f"\n1. Random Undersampling:")
print(f"   Before: {len(y_train)} → After: {len(y_rus)} samples")
print(f"   Fraud ratio: {y_rus.mean()*100:.1f}%")

# 2. Oversampling - Duplicate minority class
ros = RandomOverSampler(random_state=42)
X_ros, y_ros = ros.fit_resample(X_train_scaled, y_train)
print(f"\n2. Random Oversampling:")
print(f"   Before: {len(y_train)} → After: {len(y_ros)} samples")
print(f"   Fraud ratio: {y_ros.mean()*100:.1f}%")

# 3. SMOTE - Synthetic Minority Over-sampling
smote = SMOTE(random_state=42, k_neighbors=5)
X_smote, y_smote = smote.fit_resample(X_train_scaled, y_train)
print(f"\n3. SMOTE (Synthetic Minority Over-sampling Technique):")
print(f"   Before: {len(y_train)} → After: {len(y_smote)} samples")
print(f"   Fraud ratio: {y_smote.mean()*100:.1f}%")

# Store all datasets for comparison
sampling_strategies = {
    'Original': (X_train_scaled, y_train),
    'Undersampling': (X_rus, y_rus),
    'Oversampling': (X_ros, y_ros),
    'SMOTE': (X_smote, y_smote)
}

In [ ]:
# Visualize sampling strategies
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

titles = ['Original Distribution', 'Undersampling', 'Oversampling', 'SMOTE']
datasets = [y_train, y_rus, y_ros, y_smote]

for idx, (ax, title, data) in enumerate(zip(axes, titles, datasets)):
    counts = pd.Series(data).value_counts().sort_index()
    colors = ['#2ecc71', '#e74c3c']
    bars = ax.bar(['Legitimate', 'Fraud'], counts.values, color=colors, edgecolor='black', linewidth=1.2)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_ylabel('Count')
    ax.set_xlabel('Class')
    
    # Add count labels
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:,.0f}',
                   xy=(bar.get_x() + bar.get_width() / 2, height),
                   xytext=(0, 3),
                   textcoords="offset points",
                   ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('sampling_strategies.png', dpi=150, bbox_inches='tight')
plt.show()

### Sampling Strategy Trade-offs:

| Strategy | Pros | Cons |
|----------|------|------|
| **Undersampling** | Faster training, no data duplication | May lose important patterns |
| **Oversampling** | Preserves all original data | Risk of overfitting |
| **SMOTE** | Creates synthetic samples, better generalization | May introduce noise |

---

<a id='modeling'></a>
## 5. Model Training

In [ ]:
# Helper function to train and evaluate models
def train_and_evaluate(model, X_train, y_train, X_test, y_test, model_name):
    """
    Train model and return comprehensive evaluation metrics.
    """
    # Train
    model.fit(X_train, y_train)
    
    # Predictions
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else model.decision_function(X_test)
    
    # Metrics
    metrics = {
        'Model': model_name,
        'Precision': precision_score(y_test, y_pred, pos_label=1),
        'Recall': recall_score(y_test, y_pred, pos_label=1),
        'F1-Score': f1_score(y_test, y_pred, pos_label=1),
        'ROC-AUC': roc_auc_score(y_test, y_pred_proba),
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba
    }
    
    return model, metrics

In [ ]:
# Define base models with reasonable defaults
def get_random_forest():
    return RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=2,
        class_weight='balanced',  # Handle imbalance internally
        random_state=42,
        n_jobs=-1
    )

def get_xgboost():
    return XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        scale_pos_weight=len(y_train) / y_train.sum(),  # Handle imbalance
        random_state=42,
        n_jobs=-1,
        use_label_encoder=False,
        eval_metric='logloss'
    )

In [ ]:
# Store all results
results = []
trained_models = {}

print("\n" + "=" * 60)
print("MODEL TRAINING - ALL COMBINATIONS")
print("=" * 60)

In [ ]:
# Train models with each sampling strategy
for sampling_name, (X_sample, y_sample) in sampling_strategies.items():
    print(f"\n{'='*50}")
    print(f"Training with {sampling_name} sampling...")
    print(f"{'='*50}")
    
    # Random Forest
    print("\n[1/2] Training Random Forest...")
    rf_model = get_random_forest()
    rf_trained, rf_metrics = train_and_evaluate(
        rf_model, X_sample, y_sample, X_test_scaled, y_test,
        f'RF_{sampling_name}'
    )
    results.append(rf_metrics)
    trained_models[f'RF_{sampling_name}'] = rf_trained
    print(f"  ✓ ROC-AUC: {rf_metrics['ROC-AUC']:.4f}")
    print(f"  ✓ Recall (Fraud): {rf_metrics['Recall']:.4f}")
    
    # XGBoost
    print("\n[2/2] Training XGBoost...")
    xgb_model = get_xgboost()
    xgb_trained, xgb_metrics = train_and_evaluate(
        xgb_model, X_sample, y_sample, X_test_scaled, y_test,
        f'XGB_{sampling_name}'
    )
    results.append(xgb_metrics)
    trained_models[f'XGB_{sampling_name}'] = xgb_trained
    print(f"  ✓ ROC-AUC: {xgb_metrics['ROC-AUC']:.4f}")
    print(f"  ✓ Recall (Fraud): {xgb_metrics['Recall']:.4f}")

### Hyperparameter Tuning (Best Model)

In [ ]:
# Find best performing configuration so far
results_df = pd.DataFrame(results)
best_idx = results_df['ROC-AUC'].idxmax()
best_model_name = results_df.loc[best_idx, 'Model']
print(f"Best model so far: {best_model_name} (ROC-AUC: {results_df.loc[best_idx, 'ROC-AUC']:.4f})")

In [ ]:
# Hyperparameter tuning for XGBoost with SMOTE (typically best combo)
print("\nRunning hyperparameter tuning for XGBoost + SMOTE...")

# Define parameter grid
param_grid = {
    'max_depth': [4, 6, 8],
    'learning_rate': [0.05, 0.1, 0.2],
    'n_estimators': [50, 100, 200]
}

# Use stratified k-fold
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# Grid search
xgb_base = XGBClassifier(
    scale_pos_weight=len(y_smote) / y_smote.sum(),
    random_state=42,
    n_jobs=-1,
    use_label_encoder=False,
    eval_metric='logloss'
)

grid_search = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_smote, y_smote)

print(f"\n✓ Best Parameters: {grid_search.best_params_}")
print(f"✓ Best CV ROC-AUC: {grid_search.best_score_:.4f}")

In [ ]:
# Train final tuned model on test set
xgb_tuned = XGBClassifier(
    **grid_search.best_params_,
    scale_pos_weight=len(y_smote) / y_smote.sum(),
    random_state=42,
    n_jobs=-1,
    use_label_encoder=False,
    eval_metric='logloss'
)

xgb_tuned_trained, xgb_tuned_metrics = train_and_evaluate(
    xgb_tuned, X_smote, y_smote, X_test_scaled, y_test,
    'XGB_SMOTE_Tuned'
)

results.append(xgb_tuned_metrics)
trained_models['XGB_SMOTE_Tuned'] = xgb_tuned_trained

print(f"\n✓ Tuned XGBoost + SMOTE trained successfully!")
print(f"  ROC-AUC: {xgb_tuned_metrics['ROC-AUC']:.4f}")
print(f"  Recall: {xgb_tuned_metrics['Recall']:.4f}")

<a id='evaluation'></a>
## 6. Model Evaluation & Comparison

In [ ]:
# Create comprehensive results summary
results_summary = pd.DataFrame(results)
results_summary = results_summary[['Model', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']]
results_summary = results_summary.sort_values('ROC-AUC', ascending=False)

print("\n" + "=" * 60)
print("MODEL COMPARISON SUMMARY")
print("=" * 60)
print("\nSorted by ROC-AUC (higher is better):")
print(results_summary.to_string(index=False))

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. ROC-AUC Comparison
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(results_summary)))
bars1 = axes[0, 0].barh(results_summary['Model'], results_summary['ROC-AUC'], color=colors)
axes[0, 0].set_xlabel('ROC-AUC Score', fontsize=12)
axes[0, 0].set_title('ROC-AUC Comparison', fontsize=14, fontweight='bold')
axes[0, 0].set_xlim([0.8, 1.0])
for i, v in enumerate(results_summary['ROC-AUC']):
    axes[0, 0].text(v + 0.005, i, f'{v:.4f}', va='center', fontsize=10)

# 2. Precision-Recall Comparison
x = np.arange(len(results_summary))
width = 0.35
bars2 = axes[0, 1].bar(x - width/2, results_summary['Precision'], width, label='Precision', color='#3498db')
bars3 = axes[0, 1].bar(x + width/2, results_summary['Recall'], width, label='Recall', color='#e74c3c')
axes[0, 1].set_xlabel('Model', fontsize=12)
axes[0, 1].set_ylabel('Score', fontsize=12)
axes[0, 1].set_title('Precision vs Recall', fontsize=14, fontweight='bold')
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(results_summary['Model'], rotation=45, ha='right')
axes[0, 1].legend()
axes[0, 1].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)

# 3. F1-Score Comparison
bars4 = axes[1, 0].barh(results_summary['Model'], results_summary['F1-Score'], color='#27ae60')
axes[1, 0].set_xlabel('F1-Score', fontsize=12)
axes[1, 0].set_title('F1-Score Comparison', fontsize=14, fontweight='bold')
axes[1, 0].set_xlim([0, 1])
for i, v in enumerate(results_summary['F1-Score']):
    axes[1, 0].text(v + 0.01, i, f'{v:.4f}', va='center', fontsize=10)

# 4. Combined Metric Radar Chart (Top 5 models)
top5 = results_summary.head(5)
categories = ['Precision', 'Recall', 'F1-Score', 'ROC-AUC']
angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
angles += angles[:1]

ax_radar = fig.add_subplot(2, 2, 4, polar=True)
for idx, row in top5.iterrows():
    values = row[categories].tolist()
    values += values[:1]
    ax_radar.plot(angles, values, 'o-', linewidth=2, label=row['Model'])
    ax_radar.fill(angles, values, alpha=0.1)

ax_radar.set_xticks(angles[:-1])
ax_radar.set_xticklabels(categories, fontsize=10)
ax_radar.set_title('Top 5 Models - Multi-Metric View', fontsize=14, fontweight='bold', pad=20)
ax_radar.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=8)
ax_radar.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Confusion Matrices for Top Models
top_models = results_summary.head(4)['Model'].tolist()

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for idx, model_name in enumerate(top_models):
    model = trained_models[model_name]
    y_pred = model.predict(X_test_scaled)
    cm = confusion_matrix(y_test, y_pred)
    
    # Calculate percentages
    cm_pct = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                cbar=False, annot_kws={'size': 12})
    axes[idx].set_title(f'{model_name}\nConfusion Matrix', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Predicted', fontsize=11)
    axes[idx].set_ylabel('Actual', fontsize=11)
    axes[idx].set_xticklabels(['Legitimate', 'Fraud'])
    axes[idx].set_yticklabels(['Legitimate', 'Fraud'])

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ROC Curves for all models
plt.figure(figsize=(12, 10))

for model_name in results_summary['Model']:
    model = trained_models[model_name]
    if hasattr(model, 'predict_proba'):
        y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        y_pred_proba = model.decision_function(X_test_scaled)
    
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    
    plt.plot(fpr, tpr, label=f'{model_name} (AUC={roc_auc:.4f})', linewidth=2)

# Diagonal reference line
plt.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Random Classifier')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves - All Models', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=9)
plt.grid(alpha=0.3)
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Precision-Recall Curves
plt.figure(figsize=(12, 10))

for model_name in results_summary['Model']:
    model = trained_models[model_name]
    if hasattr(model, 'predict_proba'):
        y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        y_pred_proba = model.decision_function(X_test_scaled)
    
    precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)
    avg_precision = precision_score(y_test, model.predict(X_test_scaled), pos_label=1)
    
    plt.plot(recall, precision, label=f'{model_name}', linewidth=2)

plt.xlabel('Recall', fontsize=12)
plt.ylabel('Precision', fontsize=12)
plt.title('Precision-Recall Curves - All Models', fontsize=14, fontweight='bold')
plt.legend(loc='lower left', fontsize=9)
plt.grid(alpha=0.3)
plt.savefig('pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### Evaluation Summary:

**Key Metrics Explained:**
- **ROC-AUC**: Overall model discrimination ability (1.0 = perfect)
- **Precision**: Of all predicted fraud, how many were actually fraud?
- **Recall**: Of all actual fraud, how many did we catch?
- **F1-Score**: Harmonic mean of precision and recall

---

<a id='business'></a>
## 7. Business Interpretation & Recommendations

In [ ]:
# Get the best model
best_model_name = results_summary.iloc[0]['Model']
best_model = trained_models[best_model_name]
best_metrics = results_summary.iloc[0]

print("\n" + "=" * 60)
print("BEST PERFORMING MODEL")
print("=" * 60)
print(f"\nModel: {best_model_name}")
print(f"ROC-AUC: {best_metrics['ROC-AUC']:.4f}")
print(f"Precision: {best_metrics['Precision']:.4f}")
print(f"Recall: {best_metrics['Recall']:.4f}")
print(f"F1-Score: {best_metrics['F1-Score']:.4f}")

In [ ]:
# Business Impact Analysis
y_pred_best = best_model.predict(X_test_scaled)
cm = confusion_matrix(y_test, y_pred_best)

tn, fp, fn, tp = cm.ravel()

print("\n" + "=" * 60)
print("BUSINESS IMPACT ANALYSIS")
print("=" * 60)

print(f"\n--- Confusion Matrix Breakdown ---")
print(f"True Negatives (TN): {tn:,} - Correctly identified legitimate transactions")
print(f"False Positives (FP): {fp:,} - Legitimate transactions flagged as fraud")
print(f"False Negatives (FN): {fn:,} - Fraud transactions MISSED ⚠️")
print(f"True Positives (TP): {tp:,} - Fraud correctly detected ✓")

In [ ]:
# Cost-Benefit Analysis with hypothetical values
print("\n" + "=" * 60)
print("COST-BENEFIT ANALYSIS (Hypothetical Values)")
print("=" * 60)

# Assumptions (adjust based on your business)
AVG_FRAUD_AMOUNT = 500  # Average fraudulent transaction amount in $
COST_FALSE_POSITIVE = 10  # Cost of investigating a false alarm (customer service, etc.)
COST_FALSE_NEGATIVE = AVG_FRAUD_AMOUNT  # Full loss from missed fraud
REVENUE_TRUE_POSITIVE = AVG_FRAUD_AMOUNT  # Fraud amount saved

total_fraud_prevented = tp * AVG_FRAUD_AMOUNT
total_fraud_missed = fn * AVG_FRAUD_AMOUNT
total_fp_cost = fp * COST_FALSE_POSITIVE

net_benefit = total_fraud_prevented - total_fraud_missed - total_fp_cost

print(f"\n--- Financial Impact ---")
print(f"Fraud Prevented: ${total_fraud_prevented:,.2f}")
print(f"Fraud Missed: ${total_fraud_missed:,.2f}")
print(f"False Positive Investigation Cost: ${total_fp_cost:,.2f}")
print(f"\n💰 Net Financial Benefit: ${net_benefit:,.2f}")

print(f"\n--- Per-Transaction Economics ---")
print(f"Average fraud prevented per transaction: ${total_fraud_prevented / len(y_test):.4f}")
print(f"ROI: {(net_benefit / (total_fraud_missed + total_fp_cost)) * 100:.1f}%" if (total_fraud_missed + total_fp_cost) > 0 else "N/A")

In [ ]:
# Threshold Optimization Analysis
print("\n" + "=" * 60)
print("THRESHOLD OPTIMIZATION ANALYSIS")
print("=" * 60)

y_pred_proba_best = best_model.predict_proba(X_test_scaled)[:, 1]

# Test different thresholds
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
threshold_results = []

for thresh in thresholds:
    y_pred_thresh = (y_pred_proba_best >= thresh).astype(int)
    
    precision = precision_score(y_test, y_pred_thresh, pos_label=1, zero_division=0)
    recall = recall_score(y_test, y_pred_thresh, pos_label=1, zero_division=0)
    f1 = f1_score(y_test, y_pred_thresh, pos_label=1, zero_division=0)
    
    cm_thresh = confusion_matrix(y_test, y_pred_thresh)
    tn_t, fp_t, fn_t, tp_t = cm_thresh.ravel()
    
    # Calculate business impact at this threshold
    benefit_t = tp_t * AVG_FRAUD_AMOUNT - fn_t * AVG_FRAUD_AMOUNT - fp_t * COST_FALSE_POSITIVE
    
    threshold_results.append({
        'Threshold': thresh,
        'Precision': precision,
        'Recall': recall,
        'F1': f1,
        'FP': fp_t,
        'FN': fn_t,
        'Net_Benefit': benefit_t
    })

threshold_df = pd.DataFrame(threshold_results)
print("\nThreshold Analysis Table:")
print(threshold_df.to_string(index=False))

In [ ]:
# Visualize threshold trade-offs
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Precision-Recall Trade-off by Threshold
ax1 = axes[0, 0]
ax1.plot(threshold_df['Threshold'], threshold_df['Precision'], 'b-o', linewidth=2, label='Precision')
ax1.plot(threshold_df['Threshold'], threshold_df['Recall'], 'r-o', linewidth=2, label='Recall')
ax1.plot(threshold_df['Threshold'], threshold_df['F1'], 'g-o', linewidth=2, label='F1-Score')
ax1.set_xlabel('Decision Threshold', fontsize=12)
ax1.set_ylabel('Score', fontsize=12)
ax1.set_title('Precision-Recall Trade-off by Threshold', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(alpha=0.3)

# 2. False Positives vs False Negatives
ax2 = axes[0, 1]
ax2.plot(threshold_df['Threshold'], threshold_df['FP'], 'r-o', linewidth=2, label='False Positives')
ax2.plot(threshold_df['Threshold'], threshold_df['FN'], 'b-o', linewidth=2, label='False Negatives')
ax2.set_xlabel('Decision Threshold', fontsize=12)
ax2.set_ylabel('Count', fontsize=12)
ax2.set_title('Error Types by Threshold', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(alpha=0.3)

# 3. Net Financial Benefit
ax3 = axes[1, 0]
bars = ax3.bar(threshold_df['Threshold'], threshold_df['Net_Benefit'], color='steelblue', edgecolor='black')
ax3.set_xlabel('Decision Threshold', fontsize=12)
ax3.set_ylabel('Net Benefit ($)', fontsize=12)
ax3.set_title('Financial Impact by Threshold', fontsize=14, fontweight='bold')
ax3.axhline(y=0, color='red', linestyle='--', linewidth=2)

# Highlight best threshold
best_thresh_idx = threshold_df['Net_Benefit'].idxmax()
best_thresh = threshold_df.loc[best_thresh_idx, 'Threshold']
bars[best_thresh_idx].set_color('#27ae60')
ax3.annotate(f'Best: {best_thresh}', xy=(best_thresh, threshold_df.loc[best_thresh_idx, 'Net_Benefit']),
             xytext=(0, 10), textcoords='offset points', ha='center', fontsize=11, fontweight='bold')

# 4. Recommended Operating Zone
ax4 = axes[1, 1]
ax4.fill_between([0.2, 0.5], 0, 1, alpha=0.3, color='green', label='Recommended Zone')
ax4.plot(threshold_df['Threshold'], threshold_df['Recall'], 'r-o', linewidth=2, label='Recall (Fraud Detection)')
ax4.set_xlabel('Decision Threshold', fontsize=12)
ax4.set_ylabel('Recall', fontsize=12)
ax4.set_title('Recommended Operating Zone', fontsize=14, fontweight='bold')
ax4.set_ylim(0, 1)
ax4.legend()
ax4.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('threshold_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Final Recommendations
print("\n" + "=" * 60)
print("EXECUTIVE SUMMARY & RECOMMENDATIONS")
print("=" * 60)

print("""
┌─────────────────────────────────────────────────────────────────┐
│                    KEY FINDINGS                                 │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  1. BEST MODEL: {} │
│     - Sampling: SMOTE + XGBoost with tuned hyperparameters      │
│     - ROC-AUC: {:.4f} (excellent discrimination)                │
│     - Recall: {:.2%} (fraud detection rate)                     │
│                                                                 │
│  2. BUSINESS IMPACT:                                            │
│     - Net Financial Benefit: ${:,.2f}                           │
│     - Fraud Prevented: ${:,.2f}                                 │
│     - Recommended Threshold: {}                                 │
│                                                                 │
│  3. PRECISION vs RECALL TRADE-OFF:                              │
│     - Higher threshold → More precision, fewer false alarms     │
│     - Lower threshold → Higher recall, catch more fraud         │
│     - For fraud detection, PRIORITIZE RECALL (catch fraud)      │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
""".format(
        best_model_name,
        best_metrics['ROC-AUC'],
        best_metrics['Recall'],
        net_benefit,
        total_fraud_prevented,
        best_thresh
    ))

In [ ]:
# Deployment Recommendations
print("\n" + "=" * 60)
print("DEPLOYMENT RECOMMENDATIONS")
print("=" * 60)

print("""
┌─────────────────────────────────────────────────────────────────┐
│              PRODUCTION DEPLOYMENT CHECKLIST                    │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  ✓ MODEL SELECTION:                                             │
│    Deploy XGBoost + SMOTE with tuned hyperparameters            │
│                                                                 │
│  ✓ THRESHOLD SETTING:                                           │
│    Use threshold = {:.2f} for optimal financial outcome         │
│    Consider dynamic thresholds based on risk appetite           │
│                                                                 │
│  ✓ MONITORING:                                                  │
│    - Track precision/recall drift weekly                        │
│    - Monitor false positive rate (customer complaints)          │
│    - Alert on significant performance degradation               │
│                                                                 │
│  ✓ RETRAINING STRATEGY:                                         │
│    - Retrain monthly with new fraud patterns                    │
│    - Implement online learning for real-time adaptation         │
│    - Maintain model versioning and rollback capability          │
│                                                                 │
│  ✓ RISK MITIGATION:                                             │
│    - Implement human review for borderline cases (0.3-0.7)      │
│    - Set up escalation process for high-value transactions      │
│    - Maintain audit trail for all predictions                   │
│                                                                 │
│  ✓ COST OPTIMIZATION:                                           │
│    - False positives cost ~${} per incident                     │
│    - False negatives cost ~${} per incident                     │
│    - Optimize threshold based on actual business costs          │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
""".format(
        best_thresh,
        COST_FALSE_POSITIVE,
        COST_FALSE_NEGATIVE
    ))

In [ ]:
# Save the best model for deployment
import joblib

# Save model and scaler
joblib.dump(best_model, 'best_fraud_model.pkl')
joblib.dump(scaler, 'amount_scaler.pkl')

# Save results summary
results_summary.to_csv('model_comparison_results.csv', index=False)

print("\n✓ Model artifacts saved successfully!")
print("  - best_fraud_model.pkl")
print("  - amount_scaler.pkl")
print("  - model_comparison_results.csv")

---

## Appendix: Key Takeaways

### For Technical Teams:
1. **SMOTE + XGBoost** consistently outperforms other combinations
2. **Class weighting** is essential for imbalanced problems
3. **Threshold tuning** is as important as model selection

### For Business Stakeholders:
1. **Recall > Precision** in fraud detection (missing fraud is costlier than false alarms)
2. **Optimal threshold** depends on your specific cost structure
3. **Continuous monitoring** is critical - fraud patterns evolve

### Next Steps:
1. Validate model on holdout dataset from different time period
2. A/B test with existing fraud detection system
3. Implement real-time scoring pipeline
4. Set up model performance dashboards

---

*Note: This notebook uses the Credit Card Fraud Detection dataset (ULB). The V1-V28 features are PCA-transformed for privacy. Actual business deployment should include additional feature engineering and domain-specific validation.*